# 06 — Agent Loop Prototype

LangChain ReAct agent with custom financial tools.
Demonstrates multi-turn conversation with autonomous reasoning.

In [0]:
# Environment-aware setup: auto-installs deps only on Databricks
import os

if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'langchain', 'langchain-community', 'langgraph', 'pandas>=2.0'])
    dbutils.library.restartPython()  # noqa: F821
else:
    print("Running locally — skipping Databricks pip install. "
          "Make sure you've run: pip install -r requirements.txt")

In [0]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.agent.agent_core import init_agent, TOOLS
from src.agent.memory import SessionMemory

## 1. Initialize Agent

Configure the LLM and data path, then create the agent executor.

In [0]:
from src.env_utils import is_databricks, default_llm_model, default_data_path

# Auto-detect the right LLM model
if is_databricks():
    os.environ['LLM_MODEL'] = 'databricks/databricks-meta-llama-3-3-70b-instruct'
else:
    os.environ['LLM_MODEL'] = 'ollama/llama3.1'  # or 'openai/gpt-4'

DATA_PATH = default_data_path() if not os.path.exists('../data/virtual_financial_advisor_data.csv') else '../data/virtual_financial_advisor_data.csv'

print(f"Environment: {'Databricks' if is_databricks() else 'Local'}")
print(f"LLM Model: {default_llm_model()}")
print(f"Data Path: {DATA_PATH}")

session = SessionMemory()
agent = init_agent(data_path=DATA_PATH, session_memory=session)

print('Agent initialized with tools:', [t.name for t in TOOLS])

## 2. Multi-Turn Conversation Demo

The agent autonomously decides which tools to call based on the user's query.

In [0]:
# Turn 1: Load data & analyze spending
response = agent.invoke({'messages': [{'role': 'user', 'content': 'Load data for user_1 and analyze their spending patterns.'}]})
answer = [msg.content for msg in response['messages'] if hasattr(msg, 'type') and msg.type == 'ai' and msg.content]
print('Agent:', answer[-1] if answer else response)

In [0]:
# Turn 2: Detect risks
response = agent.invoke({'messages': [{'role': 'user', 'content': 'What financial risks do you see for this user?'}]})
answer = [msg.content for msg in response['messages'] if hasattr(msg, 'type') and msg.type == 'ai' and msg.content]
print('Agent:', answer[-1] if answer else response)

In [0]:
# Turn 3: Scenario simulation
response = agent.invoke({'messages': [{'role': 'user', 'content': 'What if I save 10% more of my income each month?'}]})
answer = [msg.content for msg in response['messages'] if hasattr(msg, 'type') and msg.type == 'ai' and msg.content]
print('Agent:', answer[-1] if answer else response)

In [0]:
# Turn 4: Personalized advice
response = agent.invoke({'messages': [{'role': 'user', 'content': 'Based on everything, give me personalized financial advice.'}]})
answer = [msg.content for msg in response['messages'] if hasattr(msg, 'type') and msg.type == 'ai' and msg.content]
print('Agent:', answer[-1] if answer else response)

## 3. Inspect Agent Reasoning

Check the session memory and agent's intermediate steps.

In [0]:
print('=== Conversation History ===')
print(session.get_history_str())

In [0]:
print('=== Cached User Profile ===')
print(session.get_user_profile())

## 4. Additional Queries

Try your own questions:

In [0]:
# Try: 'Classify my expenses', 'What if I reduce dining by 30%?', etc.
response = agent.invoke({'messages': [{'role': 'user', 'content': 'Classify my expenses and show the breakdown.'}]})
answer = [msg.content for msg in response['messages'] if hasattr(msg, 'type') and msg.type == 'ai' and msg.content]
print('Agent:', answer[-1] if answer else response)

In [0]:
# Clear session if needed
session.clear()